# Eksploracja danych
# laboratorium nr 3
# Segmentacja i reguły asocjacyjne


## I. Segmentacja

Segmentacja (klasteryzacja) służy podziałowi zbioru uczącego na kilka kategorii bez wskazania atrybutu celu. Wyniki klasteryzacji mogą być bardzo przydatne dla analityka (lub jego klienta), jako że pozwalają wykryć charakterystyczne grupy przykładów (np. pacjentów, kredytobiorców, dostawy itp.). Generalną zasadą stosowaną przy algorytmach tworzenia modeli grupujących jest dążenie do jak najmniejszych różnic w ramach grupy przykładów oraz maksymalizowanie różnic pomiędzy grupami.

### Algorytmy:
- K-Means
- EM (Gaussian Mixture)

**Dokumentacja**:
- https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html
- https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html


In [ ]:
import pandas as pd

### 1. Przygotowanie danych

Algorytmy klastrujące operują na danych liczbowych, dlatego należy zamienić wszystkie kolumny nienumeryczne na numeryczne. Z poprzednich zajęć wiemy, jak to zrobić. Należy również usunąć atrybut klasy.


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

print(df.head())

### 2. Algorytm EM (Gaussian Mixture)

Poniższy przykład pokazuje, w jaki sposób można wykorzystać algorytm EM do klasteryzacji klientów.

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=3, random_state=42)
gmm.fit(df)

labels_gmm = gmm.predict(df)
labels_gmm[:10]


Wizualizacja klastrów.

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df.iloc[:,0], df.iloc[:,1], c=labels_gmm)
plt.title("EM")
plt.show()

### 2. Algorytm K-MEANS

Poniższy przykład pokazuje, w jaki sposób można wykorzystać algorytm K-Means do klasteryzacji klientów. Dane zostały już wcześniej przygotowane.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(df)

labels = kmeans.labels_
labels[:10]


#### Zadanie 1. (1 pkt)
Zwizualizuj wynik klasteryzacji dla algorytmu K-Means. Czy wynik różni się od tego, który wygenerował algorytm EM?

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df.iloc[:,0], df.iloc[:,1], c=labels)
plt.title("K-Means")
plt.show()

#### Zadanie 2. (1 pkt)
Sprawdź, które punkty zostały inaczej sklasyfikowane i porównaj wyniki z klasami z *iris.target*.

In [ ]:
#Miejsce na rozwiązanie



#### Zadanie 3. (2 pkt)
Zmień liczbę klastrów (2, 4, 5) dla obu algorytmów. Czy coś się zmieniło?


In [ ]:
#Miejsce na rozwiązanie


Wykonamy teraz normalizację danych z wykorzystaniem klasy MinMaxScaler.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df)

Powtórzymy segmentację algorytmem K-Means.

In [ ]:
kmeans_scaled = KMeans(n_clusters=3, random_state=42)
kmeans_scaled.fit(X_scaled)

labels_scaled = kmeans_scaled.labels_

#### Zadanie 4. (1 pkt)
Zwizualizuj wynik klasteryzacji dla algorytmu K-Means po normalizacji. Czy wynik różni się od tego dla danych przed normalizacją?

#### **Klasteryzacja dla danych *churn***
Na początek wczytamy plik *churn.txt* (jak pamiętasz, zawiera on dane na temat klientów firmy telekomunikacyjnej i ich rezygnacji z usług).

In [ ]:
churn = pd.read_csv("churn.txt")
churn.columns = churn.columns.str.strip()
churn.head()

Zamiana zmiennych kategorycznych - w tym wypadku nazw stanów - na zmienną numeryczną.

In [ ]:
churn['State'] = pd.factorize(churn['State'])[0]
churn

#### Zadanie 5. (1 pkt)
Przygotuj dane do klasteryzacji (zamień 'yes'/'no' na liczby oraz usuń wszystkie zbędne kolumny) i obejrzyj je.

In [ ]:
#Miejsce na rozwiązanie

#### Zadanie 6. (1,5 pkt)
Przeprowadź klasteryzację algorytmem k-średnich. Przetestuj różną liczbę klastrów (2,3,4). Sprawdź rozkład zmiennych *Int'l Plan, Vmail Plan* pomiędzy segmentami na wizualizacji. Co możesz powiedzieć o wynikach?

In [ ]:
#Miejsce na odpowiedź

## II. Reguły asocjacyjne

Reguły asocjacyjne służą odnajdywaniu zależności pomiędzy wartościami poszczególnych atrybutów bez wskazywania konkretnego atrybutu celu. Reguły asocjacyjne są przydatne dla analityka (lub jego klienta), ze względu na to, że pozwalają na określenie pewnych charakterystycznych prawidłowości zachodzących w zbiorze przykładów. Reguły asocjacji są także często stosowane jako narzędzie wspierające EDA (eksploracyjną analizę danych). 

#### Zainstaluj bibliotekę!: **pip install mlxtend**

**Dokumentacja**:
- https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/
- https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/


In [ ]:
#pip install liac-arff pandas
import arff
import pandas as pd

# 1. Wczytaj plik ARFF
with open('hepatitis.arff', 'r') as f:
    dataset = arff.load(f)

# 2. Wyodrębnij nazwy danych i atrybutów
data = dataset['data']
attributes = [attr[0] for attr in dataset['attributes']]

# 3. Utwórz pandas DataFrame
df = pd.DataFrame(data, columns=attributes)

# Wyświetl pierwsze kilka wierszy
print(df.head())


Będziemy wykorzystywać algorytm Apriori, zatem ze zbioru należy usunąć atrybuty numeryczne (nienominalne). Można również przekształcić te atrybuty na nominalne.

In [ ]:
df_trans = df.drop(columns=['AGE','BILIRUBIN','ALK_PHOSPHATE','SGOT','ALBUMIN','PROTIME'])
print(df_trans.head())

Dodatkowo musimy usunąć bądź uzupełnić wartości brakujące.

In [ ]:
df_trans.fillna({'SEX': df_trans['SEX'].mode()[0],'STEROID': df_trans['STEROID'].mode()[0],'FATIGUE': df_trans['FATIGUE'].mode()[0],'MALAISE': df_trans['MALAISE'].mode()[0],
                 'ANOREXIA': df_trans['ANOREXIA'].mode()[0],'LIVER_BIG': df_trans['LIVER_BIG'].mode()[0],'LIVER_FIRM': df_trans['LIVER_FIRM'].mode()[0],
                 'SPLEEN_PALPABLE': df_trans['SPLEEN_PALPABLE'].mode()[0],'SPIDERS': df_trans['SPIDERS'].mode()[0],'ASCITES': df_trans['ASCITES'].mode()[0],
                 'VARICES': df_trans['VARICES'].mode()[0]}, inplace=True)
df_trans.isnull().sum()

Algorytm Apriori działa na typach *boolean* i *binary*. Musimy więc odpowiednio przekształcić pozostałe wartości. 

In [ ]:
df_trans=df_trans.replace({'SEX': {'male': 1, 
                                'female': 0}})
df_trans=df_trans.replace({'Class': {'LIVE': 1, 'DIE': 0}})
df_trans = df_trans.replace({'yes': True, 
                                'no': False})
df_trans.head(10)

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

#Wywołujemy algorytm Apriori z minimalną wartością support=0.2
frequent_items = apriori(df_trans, min_support=0.2, use_colnames=True)
frequent_items


#### Zadanie 7. (0,5 pkt)
Przetestuj różne wartości parametru *min_support*. Co się zmienia?

In [ ]:
#Miejsce na rozwiązanie

In [ ]:
#Generujemy reguły asocjacyjne na podstawie zbiorów wygenerowanych przez algorytm Apriori
rules = association_rules(frequent_items, metric="confidence", min_threshold=0.6)
rules


#### Zadanie 8. (1 pkt)
Przetestuj różne wartości dla parametru *confidence* i zwróć 10 najlepszych reguł. Zinterpretuj trzy pierwsze z nich.

In [ ]:
#Miejsce na rozwiązanie

#### Zadanie 8. (1 pkt)
Przetestuj różne miary dla parametru *metric* (support, lift) i różną wartość dla threshold. Zwróć 10 najlepszych reguł dla każdego przypadku. Co możesz powiedzieć o wynikach?

In [ ]:
#Miejsce na rozwiązanie